# Walmart Sales Analysis: Python + SQL Server

End-to-end project: download, clean and load Walmart sales data, then analyse it in SQL Server.

## Part 1-2: Environment and data download (done in the terminal)
Environment: Python 3.12 virtual environment `my_env1`, libraries `pandas`, `sqlalchemy`, `pyodbc`, `kaggle`, `ipykernel`, `six`.
Data: `kaggle datasets download -d najir0123/walmart-10k-sales-datasets`, then `unzip` to get `Walmart.csv`.

## Part 3: Load and explore the data

In [242]:
# Import pandas, the library used to work with tables (DataFrames)
import pandas as pd

In [243]:
# Load the CSV file into a DataFrame called df
# Capital W: file names are case-sensitive on Linux
df = pd.read_csv('Walmart.csv')

# Shape = (number of rows, number of columns): expected (10051, 11)
df.shape

(10051, 11)

In [244]:
# First 5 rows: quick look at the data
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
0,1,WALM003,San Antonio,Health and beauty,$74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48
1,2,WALM048,Harlingen,Electronic accessories,$15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48
2,3,WALM067,Haltom City,Home and lifestyle,$46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33
3,4,WALM064,Bedford,Health and beauty,$58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33
4,5,WALM013,Irving,Sports and travel,$86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48


In [245]:
# Column types and number of non-null values per column
# unit_price is "object" (text) because of the "$" sign
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      10051 non-null  int64  
 1   Branch          10051 non-null  str    
 2   City            10051 non-null  str    
 3   category        10051 non-null  str    
 4   unit_price      10020 non-null  str    
 5   quantity        10020 non-null  float64
 6   date            10051 non-null  str    
 7   time            10051 non-null  str    
 8   payment_method  10051 non-null  str    
 9   rating          10051 non-null  float64
 10  profit_margin   10051 non-null  float64
dtypes: float64(3), int64(1), str(7)
memory usage: 863.9 KB


In [246]:
# Number of fully duplicated rows: expected 51
df.duplicated().sum()

np.int64(51)

In [247]:
# Number of missing values per column: expected 31 in unit_price and quantity
df.isnull().sum()

invoice_id         0
Branch             0
City               0
category           0
unit_price        31
quantity          31
date               0
time               0
payment_method     0
rating             0
profit_margin      0
dtype: int64

**Findings:** 51 duplicated rows, 31 rows with a missing `unit_price` / `quantity`, and `unit_price` stored as text because of the `$` sign. These three issues are fixed in Part 4.

## Part 4: Clean the data

In [248]:
# 1. Remove duplicated rows (identical lines)
df.drop_duplicates(inplace=True)   # inplace=True: modifies df directly
df.duplicated().sum()              # check: should be 0

/var/folders/vb/b4pxrh3j6_1cq9fvp7rc8pqr0000gn/T/ipykernel_12665/1658097466.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop_duplicates(inplace=True)   # inplace=True: modifies df directly


np.int64(0)

In [249]:
# Shape after removing duplicates: expected (10000, 11)
df.shape

(10000, 11)

In [250]:
# 2. Remove rows with missing values
# We drop instead of filling: without price or quantity, the total cannot be computed
df.dropna(inplace=True)
df.isnull().sum()                  # check: all columns at 0

/var/folders/vb/b4pxrh3j6_1cq9fvp7rc8pqr0000gn/T/ipykernel_12665/603325286.py:3: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.dropna(inplace=True)


invoice_id        0
Branch            0
City              0
category          0
unit_price        0
quantity          0
date              0
time              0
payment_method    0
rating            0
profit_margin     0
dtype: int64

In [251]:
# Shape after removing missing values: expected (9969, 11)
df.shape

(9969, 11)

In [252]:
# 3. Convert unit_price from text ("$74.69") to a number (74.69)
# The "$" must be removed first, otherwise Python cannot convert it
df['unit_price'] = df['unit_price'].str.replace('$', '', regex=False).astype(float)
df.dtypes                          # check: unit_price should now be float64

invoice_id          int64
Branch                str
City                  str
category              str
unit_price        float64
quantity          float64
date                  str
time                  str
payment_method        str
rating            float64
profit_margin     float64
dtype: object

In [253]:
# 4. Create a new column: total = unit price x quantity
df['total'] = df['unit_price'] * df['quantity']
df.head()                          # check: first row total = 74.69 x 7 = 522.83

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48,522.83
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48,76.40
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33,324.31
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33,465.76
4,5,WALM013,Irving,Sports and travel,86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48,604.17


In [254]:
# 5. Put column names in lower case (Branch -> branch, City -> city)
# Avoids case issues between tools, especially with PostgreSQL
df.columns = df.columns.str.lower()
df.columns

Index(['invoice_id', 'branch', 'city', 'category', 'unit_price', 'quantity',
       'date', 'time', 'payment_method', 'rating', 'profit_margin', 'total'],
      dtype='str')

In [255]:
# 6. Save the cleaned data to a new CSV file (index=False: no row-number column)
df.to_csv('walmart_clean_data.csv', index=False)
df.shape                           # final shape: expected (9969, 12)

(9969, 12)

## Part 5: Export the clean data to SQL Server
The database `walmart_db` is created first in SQL (see `walmart.sql`). The table `walmart` is created by pandas.

In [256]:
# create_engine: builds the "engine", the object that talks to SQL Server
# text: lets us send a small hand-written SQL query (used for the connection test)
from sqlalchemy import create_engine, text

# quote_plus: makes a text safe to put inside an address (password with "@" or ";")
from urllib.parse import quote_plus

# getpass: asks for the password in a pop-up, without showing it or saving it in the notebook
from getpass import getpass

In [257]:
# --- Connection settings for SQL Server running in Docker ---
server = '127.0.0.1'      # your own Mac, forced to IPv4
port = 1433               # default SQL Server port
database = 'walmart_db'   # the database created in SQL
user = 'sa'               # default admin user (the tutor used root / postgres)

# Pop-up asking for the password: never shown or saved in the notebook
password = getpass('SQL Server password: ')

# --- Engine: same idea as the tutor's create_engine("mysql+pymysql://...") ---
# mssql+pymssql = SQL Server through pymssql
engine = create_engine(
    f"mssql+pymssql://{user}:{quote_plus(password)}@{server}:{port}/{database}"
)

In [258]:
# Open a real connection to SQL Server ("with" closes it automatically afterwards)
with engine.connect() as conn:
    # Ask the server for its version: if this prints, the connection works
    print(conn.execute(text("SELECT @@VERSION")).scalar())

Microsoft SQL Server 2025 (RTM-CU7) (KB5096981) - 17.0.4065.4 (X64) 
	Jul  8 2026 23:26:08 
	Copyright (C) 2025 Microsoft Corporation
	Enterprise Developer Edition (64-bit) on Linux (Ubuntu 24.04.4 LTS) <X64>


In [259]:
# Export the DataFrame to SQL Server: creates the table "walmart" and inserts all rows
# if_exists='replace': drops and recreates the table, so re-running never duplicates rows
# index=False: do not export the row-number column
# Note: with pymssql the returned number can be wrong (51 here); check the real count in SQL
df.to_sql(name='walmart', con=engine, if_exists='replace', index=False)

51